# 05 Cross-Domain Intelligence Synthesis

Integrates evidence from domains 01-04. **Sources:** `analytics.fact_orders`, `analytics.fact_reviews`, `analytics.fact_delivery`, `staging.orders`+`staging.customers`. Metrics follow approved KPI definitions/filters in `docs/olist/03_kpi_definitions.md`.

In [1]:
from src.db import get_connection

conn = get_connection(read_only=True)


In [2]:
# Approved KPIs (overall)
overall = conn.execute("""
WITH rep AS (
  SELECT cu.customer_unique_id, COUNT(DISTINCT o.order_id) cnt
  FROM staging.orders o JOIN staging.customers cu ON o.customer_id=cu.customer_id
  GROUP BY 1
)
SELECT
  (SELECT COUNT(DISTINCT order_id) FROM analytics.fact_orders) AS order_count,
  (SELECT SUM(order_total_value) FROM analytics.fact_orders) AS gmv_payments,
  (SELECT AVG(order_total_value) FROM analytics.fact_orders) AS aov,
  (SELECT 1.0*SUM(CASE WHEN cnt>1 THEN 1 ELSE 0 END)/COUNT(*) FROM rep) AS repeat_customer_rate,
  (SELECT 1.0*SUM(CASE WHEN order_status='canceled' THEN 1 ELSE 0 END)/COUNT(*) FROM analytics.fact_orders) AS cancellation_rate,
  (SELECT AVG(delivery_time_days) FROM analytics.fact_delivery WHERE is_delivered AND delivery_time_days IS NOT NULL) AS avg_delivery_days,
  (SELECT 1.0*SUM(CASE WHEN delivery_gap_days>=0 THEN 1 ELSE 0 END)/COUNT(*) FROM analytics.fact_delivery WHERE is_delivered AND delivery_gap_days IS NOT NULL) AS on_time_delivery_rate,
  (SELECT AVG(valid_review_score) FROM analytics.fact_reviews WHERE valid_review_score IS NOT NULL) AS avg_review_score
""").fetchdf()
overall

,order_count,gmv_payments,aov,repeat_customer_rate,cancellation_rate,avg_delivery_days,on_time_delivery_rate,avg_review_score
0,99441,1.600887e+07,160.988648,0.031188,0.006285,12.497336,0.932263,4.088802


Key synthesized findings (see `outputs/findings/analysis_findings.json`, `outputs/reports/`):
- Repeat purchase rate is ~3.12% at `customer_unique_id` grain over the observed period (all statuses; identity linkage as in source).
- Delivery timeliness associates with review scores: on-time/early ~4.29 vs late ~2.27; longer delivery buckets trend lower.
- Revenue and volume concentrate in SP/RJ/MG; several remote states show longer average delivery times.
- Category performance varies by revenue and average review; interpretations must preserve item/order grain and stated filters.